In [1]:
# Python Standard Library
import sys
from os.path import join
from importlib import reload

# Community Modules
import numpy as np
from scipy import stats
import pandas as pd
from pandas.testing import assert_frame_equal
import keras

# My Modules
sys.path.insert(0, "../code")
from unpack_dataset import unpack_dataset
import model_abcsn_pretrain

sys.path.insert(0, "../../ABC-SN/code")
import abcsn_training_withSNR

from abcsn_training import get_masked_spectrum  # Potentially put this in its own script?

2026-03-16 20:36:54.986673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-16 20:36:55.005230: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-16 20:36:55.012398: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-16 20:36:55.028057: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-16 20:36:56.625354: W tensorflow/compiler/tf2

In [2]:
rng = np.random.default_rng(1415)

In [3]:
new_SNR = 100

In [4]:
DIR_DATA = "../data/forSNR/train_test_split"
DIR_MODEL = "/lustre/lrspec/users/2649/ABCSN_SNR/dev"

## 1. Load training and testing sets

In [5]:
file_dataset_trn = join(DIR_DATA, "dataset_trn.parquet")
file_signal_trn = join(DIR_DATA, "signal_trn.parquet")
file_noise_trn = join(DIR_DATA, "noise_trn.parquet")

file_dataset_tst = join(DIR_DATA, "dataset_tst.parquet")
file_signal_tst = join(DIR_DATA, "signal_tst.parquet")
file_noise_tst = join(DIR_DATA, "noise_tst.parquet")

In [6]:
df_dataset_trn = pd.read_parquet(file_dataset_trn)
df_signal_trn = pd.read_parquet(file_signal_trn)
df_noise_trn = pd.read_parquet(file_noise_trn)
df_dataset_tst = pd.read_parquet(file_dataset_tst)
df_signal_tst = pd.read_parquet(file_signal_tst)
df_noise_tst = pd.read_parquet(file_noise_tst)

In [7]:
wvl, spectra_dataset_trn, df_meta_trn = unpack_dataset(df_dataset_trn)
_1_wvl, spectra_signal_trn, df_meta_signal_trn = unpack_dataset(df_signal_trn)
_2_wvl, spectra_noise_trn, df_meta_noise_trn = unpack_dataset(df_noise_trn)
assert np.all(wvl == _1_wvl)
assert np.all(wvl == _2_wvl)
del _1_wvl, _2_wvl
assert_frame_equal(df_meta_trn, df_meta_signal_trn)
assert_frame_equal(df_meta_trn, df_meta_noise_trn)
del df_meta_signal_trn, df_meta_noise_trn

_0_wvl, spectra_dataset_tst, df_meta_tst = unpack_dataset(df_dataset_tst)
_1_wvl, spectra_signal_tst, df_meta_signal_tst = unpack_dataset(df_signal_tst)
_2_wvl, spectra_noise_tst, df_meta_noise_tst = unpack_dataset(df_noise_tst)
assert np.all(wvl == _0_wvl)
assert np.all(wvl == _1_wvl)
assert np.all(wvl == _2_wvl)
del _0_wvl, _1_wvl, _2_wvl
assert_frame_equal(df_meta_tst, df_meta_signal_tst)
assert_frame_equal(df_meta_tst, df_meta_noise_tst)
del df_meta_signal_tst, df_meta_noise_tst

assert wvl.ndim == 1, (wvl.ndim, wvl.shape)
num_wvl = wvl.size
num_spectra_trn = spectra_dataset_trn.shape[0]
num_spectra_tst = spectra_dataset_tst.shape[0]

## 2. Remove the 0-padding

This is a departure from ABC-SN but I'm going to clip the arrays to (4500, 7000) angstroms.

In [8]:
wvl_mask = wvl >= 4500
wvl_mask &= wvl <= 7000

In [9]:
wvl = wvl[wvl_mask]

spectra_dataset_trn = spectra_dataset_trn[:, wvl_mask]
spectra_signal_trn = spectra_signal_trn[:, wvl_mask]
spectra_noise_trn = spectra_noise_trn[:, wvl_mask]

spectra_dataset_tst = spectra_dataset_tst[:, wvl_mask]
spectra_signal_tst = spectra_signal_tst[:, wvl_mask]
spectra_noise_tst = spectra_noise_tst[:, wvl_mask]

num_wvl = wvl.size
num_spectra_trn = spectra_dataset_trn.shape[0]
num_spectra_tst = spectra_dataset_tst.shape[0]

Consider having a print statement here that shows the train/test distribution. number of sne/spectra per class in the training/testing sets.

## 3. Add noise corresponding to the chosen SNR

In [10]:
def standardize_SNR(new_SNR, S, signal, num_spectra, num_wvl, rng):
    new_N = S / new_SNR
    new_noise = stats.norm.rvs(
        loc=0,
        scale=np.full((num_spectra, num_wvl), new_N[..., np.newaxis]),
        random_state=rng)
    new_spectra = signal + new_noise
    return new_spectra

In [11]:
Xtrn = standardize_SNR(
    new_SNR,
    df_meta_trn["S (SNR)"].to_numpy(),
    spectra_signal_trn,
    num_spectra_trn,
    num_wvl, 
    rng)

Xtst = standardize_SNR(
    new_SNR,
    df_meta_tst["S (SNR)"].to_numpy(),
    spectra_signal_tst,
    num_spectra_tst,
    num_wvl, 
    rng)

## 4. Mask spectra for pre-training

In [12]:
mask_frac = 0.15
mask_val = 0

In [13]:
nonzero_bool = np.full(num_wvl, True)
Xtrn_masked = get_masked_spectrum(Xtrn, wvl, nonzero_bool, mask_frac, mask_val)
Xtst_masked = get_masked_spectrum(Xtst, wvl, nonzero_bool, mask_frac, mask_val)

## 5. Initialize the pre-training model

In [14]:
reload(model_abcsn_pretrain)
model_pretrain = model_abcsn_pretrain.make_model(num_wvl)
model_pretrain.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 327)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 327)    │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1, 1024)   │    335,872 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 64, 16)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64, 128)   │      2,176 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sine_position_enco… │ (None, 64, 128)   │          0 │ dense_1[0][0]     │
│ (SinePositionEncod… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64, 64)    │      8,256 │ sine_position_en… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64, 128)   │      8,320 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 128)   │          0 │ dense_1[0][0],    │
│                     │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, 64, 128)   │    132,480 │ add[0][0]         │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 128)   │    132,480 │ transformer_enco… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 128)   │    132,480 │ transformer_enco… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 128)   │    132,480 │ transformer_enco… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 128)   │    132,480 │ transformer_enco… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 128)   │    132,480 │ transformer_enco… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 8192)      │          0 │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 327)       │  2,678,784 │ reshape_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,828,288 (14.60 MB)

 Trainable params: 3,828,288 (14.60 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Compile and fit the pre-training model

In [15]:
# Initialize the masked self-attention model
loss = keras.losses.MeanSquaredError(name="mse")
optimizer = keras.optimizers.Nadam(learning_rate=1e-4)
model_pretrain.compile(loss=loss, optimizer=optimizer)

# Train the masked self-attention model
callbacks_pretrain = abcsn_training_withSNR.get_callbacks(
    25,
    10,
    0.5,
    1e-7,
    0.0005,
    join(DIR_MODEL, "pretrain_log.csv"),
)
history_pretrain = model_pretrain.fit(
    Xtrn_masked,
    Xtrn,
    validation_data=(Xtst_masked, Xtst),
    epochs=10_000,
    batch_size=64,
    callbacks=callbacks_pretrain,
)

model_pretrain.save(join(DIR_MODEL, "ABCSN_pretrain.keras"))

Epoch 1/10000
27/27 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - loss: 7.1579 - val_loss: 0.4301 - learning_rate: 1.0000e-04
Epoch 2/10000
27/27 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - loss: 3.6907 - val_loss: 0.2056 - learning_rate: 1.0000e-04
Epoch 3/10000
 8/27 ━━━━━━━━━━━━━━━━━━━━ 16s 853ms/step - loss: 2.2572

KeyboardInterrupt: 